# MODEL ARENA - Track 2: Steel Plant Energy Consumption Forecasting
Goal: predict `next_interval_Usage_kWh` (electricity use in the next 15-minute interval).
Metrics: RMSE and MAE on a chronological hold-out.

Pipeline: load data -> EDA -> persistence baseline -> Random Forest (chronological validation) -> final model -> inference function.

In [1]:
import os
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Works in Colab (after unzipping the participant bundle) and locally
CANDIDATES = ["/content/track2_steel_energy/train.csv", "track2_steel_energy/train.csv", "train.csv"]
DATA_PATH = next(p for p in CANDIDATES if os.path.exists(p))
MODEL_PATH = "track2_energy_model.joblib"

df = pd.read_csv(DATA_PATH)
print("scikit-learn", sklearn.__version__)
print(df.shape)
df.head()

scikit-learn 1.8.0
(28031, 12)


,date,Usage_kWh,Lagging_Current_Reactive.Power_kVarh,Leading_Current_Reactive_Power_kVarh,CO2(tCO2),Lagging_Current_Power_Factor,Leading_Current_Power_Factor,NSM,WeekStatus,Day_of_week,Load_Type,next_interval_Usage_kWh
0,01/01/2018 00:15,3.17,2.95,0.0,0.0,73.21,100.0,900,Weekday,Monday,Light_Load,4.00
1,01/01/2018 00:30,4.00,4.46,0.0,0.0,66.77,100.0,1800,Weekday,Monday,Light_Load,3.24
2,01/01/2018 00:45,3.24,3.28,0.0,0.0,70.28,100.0,2700,Weekday,Monday,Light_Load,3.31
3,01/01/2018 01:00,3.31,3.56,0.0,0.0,68.09,100.0,3600,Weekday,Monday,Light_Load,3.82
4,01/01/2018 01:15,3.82,4.50,0.0,0.0,64.72,100.0,4500,Weekday,Monday,Light_Load,3.28


## 1. Exploratory data analysis

In [2]:
print("Missing values per column:"); print(df.isna().sum().sum(), "total")
dates = pd.to_datetime(df["date"], format="%d/%m/%Y %H:%M")
print("Date range:", dates.min(), "->", dates.max())

# Target is exactly the NEXT row's Usage_kWh -> rows are sequential 15-min records
print("target == next row's Usage_kWh:",
      (df["next_interval_Usage_kWh"].iloc[:-1].values == df["Usage_kWh"].iloc[1:].values).all())

# Data-quality quirk: each day's NSM=0 (midnight) row is stored at the END of that day's block
# but carries the day's own date, so sorting by `date` would scramble the sequence.
print("Rows with NSM == 0:", (df["NSM"] == 0).sum())
print(df.loc[93:97, ["date", "NSM", "Usage_kWh", "next_interval_Usage_kWh"]])

print("\nLoad_Type counts:"); print(df["Load_Type"].value_counts())
print("\nCorrelation of Usage_kWh with target:", round(df["Usage_kWh"].corr(df["next_interval_Usage_kWh"]), 3))
print("Mean usage by Load_Type:"); print(df.groupby("Load_Type")["Usage_kWh"].mean().round(2))

Missing values per column:
0 total
Date range: 2018-01-01 00:00:00 -> 2018-10-19 23:45:00
target == next row's Usage_kWh: True
Rows with NSM == 0: 291
                date    NSM  Usage_kWh  next_interval_Usage_kWh
93  01/01/2018 23:30  84600       3.24                     3.67
94  01/01/2018 23:45  85500       3.67                     3.42
95  01/01/2018 00:00      0       3.42                     3.20
96  02/01/2018 00:15    900       3.20                     3.85
97  02/01/2018 00:30   1800       3.85                     3.28

Load_Type counts:
Load_Type
Light_Load      14367
Medium_Load      7808
Maximum_Load     5856
Name: count, dtype: int64

Correlation of Usage_kWh with target: 0.911
Mean usage by Load_Type:
Load_Type
Light_Load       8.90
Maximum_Load    62.50
Medium_Load     36.99
Name: Usage_kWh, dtype: float64


Key findings: no missing values; 28,031 sequential 15-minute rows (Jan-Oct 2018); strong autocorrelation (~0.91);
usage differs strongly by `Load_Type`; and the midnight rows sit at the end of each day's block, so **we keep the original row order and never sort by `date`.**

## 2. Baseline (persistence): predict next usage = current usage
Chronological split: first 80% train, last 20% validation.

In [3]:
split = int(len(df) * 0.8)
valid = df.iloc[split:]

actual = valid["next_interval_Usage_kWh"]
predicted = valid["Usage_kWh"]

print("Validation rows:", len(valid))
print("Baseline MAE:", round(mean_absolute_error(actual, predicted), 2))
print("Baseline RMSE:", round(mean_squared_error(actual, predicted) ** 0.5, 2))

Validation rows: 5607
Baseline MAE: 5.28
Baseline RMSE: 11.94


## 3. Random Forest
Features: current telemetry + `hour` + one-hot (`WeekStatus`, `Day_of_week`, `Load_Type`).
`month` is deliberately **not** used: training covers only Jan-Oct, so a later test period would contain unseen months.

In [4]:
def build_features(input_df):
    """Shared feature preparation for training and inference."""
    dates = pd.to_datetime(input_df["date"], format="%d/%m/%Y %H:%M")
    X = input_df.drop(columns=["date", "next_interval_Usage_kWh"], errors="ignore").copy()
    X["hour"] = dates.dt.hour
    return pd.get_dummies(X, columns=["WeekStatus", "Day_of_week", "Load_Type"], dtype=int)

X = build_features(df)
y = df["next_interval_Usage_kWh"]

X_train, X_valid = X.iloc[:split], X.iloc[split:]
y_train, y_valid = y.iloc[:split], y.iloc[split:]

model = RandomForestRegressor(n_estimators=150, min_samples_leaf=5, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
predicted = model.predict(X_valid)

print("Random Forest MAE:", round(mean_absolute_error(y_valid, predicted), 2))
print("Random Forest RMSE:", round(mean_squared_error(y_valid, predicted) ** 0.5, 2))

Random Forest MAE: 4.14
Random Forest RMSE: 9.09


In [5]:
importance = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
display(importance.head(10))

Usage_kWh                               0.896894
NSM                                     0.039696
hour                                    0.017143
Lagging_Current_Power_Factor            0.014906
Lagging_Current_Reactive.Power_kVarh    0.011856
Leading_Current_Reactive_Power_kVarh    0.005380
Leading_Current_Power_Factor            0.003805
Load_Type_Maximum_Load                  0.001494
Load_Type_Medium_Load                   0.001104
Day_of_week_Thursday                    0.001074
dtype: float64

## 4. Error analysis vs baseline

In [6]:
results = pd.DataFrame({"actual": y_valid, "predicted": predicted})
results["absolute_error"] = (results["actual"] - results["predicted"]).abs()
results["baseline_error"] = (df.loc[results.index, "Usage_kWh"] - results["actual"]).abs()
results["usage_band"] = pd.qcut(results["actual"], q=4, duplicates="drop")
results["Load_Type"] = df.loc[results.index, "Load_Type"]

display(results.groupby("usage_band", observed=True).agg(
    rows=("actual", "size"),
    random_forest_MAE=("absolute_error", "mean"),
    baseline_MAE=("baseline_error", "mean")).round(2))

display(results.groupby("Load_Type").agg(
    rows=("actual", "size"),
    random_forest_MAE=("absolute_error", "mean"),
    baseline_MAE=("baseline_error", "mean")).round(2))

,rows,random_forest_MAE,baseline_MAE
usage_band,,,
"(2.4490000000000003, 2.84]",1441,0.16,0.10
"(2.84, 3.6]",1418,0.26,0.26
"(3.6, 45.54]",1349,5.72,7.58
"(45.54, 138.02]",1399,10.67,13.47


,rows,random_forest_MAE,baseline_MAE
Load_Type,,,
Light_Load,2883,1.21,1.62
Maximum_Load,1164,9.87,11.98
Medium_Load,1560,5.29,7.04


## 5. Final model (trained on all rows) and saved artifact

In [7]:
final_model = RandomForestRegressor(n_estimators=150, min_samples_leaf=5, random_state=42, n_jobs=-1)
final_model.fit(X, y)

joblib.dump({"model": final_model, "columns": X.columns.tolist(), "sklearn_version": sklearn.__version__},
            MODEL_PATH, compress=3)
print("Saved", MODEL_PATH, round(os.path.getsize(MODEL_PATH) / 1e6, 1), "MB")

Saved track2_energy_model.joblib 19.1 MB


## 6. Inference function
Accepts a dataframe with the same columns as `train.csv` (the target column is optional) and returns predictions.

In [8]:
saved = joblib.load(MODEL_PATH)
final_model = saved["model"]
feature_columns = saved["columns"]

def predict_next_usage(input_df):
    features = build_features(input_df)
    features = features.reindex(columns=feature_columns, fill_value=0)  # match training columns
    return final_model.predict(features)

example_input = df.drop(columns=["next_interval_Usage_kWh"]).tail(5)
print(predict_next_usage(example_input))

[57.54583071 54.69530273 51.69774851 64.64909215 81.46892858]


In [9]:
# Optional: download the artifact when running in Google Colab
try:
    from google.colab import files
    files.download(MODEL_PATH)
except ImportError:
    pass